In [1]:
import sys
import shap
import joblib
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
from credit_risk.features import load_features

2026-07-20 18:58:49.663 | INFO     | credit_risk.config:<module>:11 - PROJ_ROOT path is: /Users/ak007/SML/Credit-Risk-Default-Prediction-System


In [3]:
cwd = Path.cwd()
project_root = cwd.parent
if project_root not in sys.path:
    sys.path.insert(0, str(project_root))
    print("Done!")

Done!


In [4]:
feature_splits = load_features()
val_df = feature_splits['val'][0]

2026-07-20 18:58:49.692 | INFO     | credit_risk.features:load_features:263 - Loading the processed features...
2026-07-20 18:58:50.069 | INFO     | credit_risk.features:load_features:270 - Loaded successfully!


In [5]:
model = joblib.load(project_root / "models" / "tuned_xgb" / "model.pkl")

In [6]:
explainer = shap.TreeExplainer(model=model[1])

In [7]:
val_df.head(10)

,num__loan_amnt,num__installment,num__annual_inc,num__dti,num__delinq_2yrs,num__inq_last_6mths,num__mths_since_last_delinq,num__mths_since_last_record,num__open_acc,num__pub_rec,...,cat__addr_state_TX,cat__addr_state_UT,cat__addr_state_VA,cat__addr_state_VT,cat__addr_state_WA,cat__addr_state_WI,cat__addr_state_WV,cat__addr_state_WY,cat__initial_list_status_f,cat__initial_list_status_w
0,0.686294,0.953623,0.486334,-0.813661,-0.356983,0.178788,-0.096518,3.072199,-0.638892,1.643191,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,1.289784,0.432154,0.540914,0.191358,-0.356983,0.178788,-0.096518,0.020373,-0.839423,-0.314275,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,0.399637,0.664406,-0.332360,2.667600,-0.356983,-0.737261,1.849699,0.020373,1.366418,-0.314275,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,-1.124174,-1.135577,-0.013979,-1.111727,3.406032,2.010887,-1.841403,0.020373,0.163232,-0.314275,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,-0.822429,-0.751283,0.304402,0.558208,-0.356983,0.178788,-0.096518,0.020373,0.163232,-0.314275,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
5,-1.003476,-0.973432,-0.241394,-0.883719,-0.356983,-0.737261,-0.096518,0.020373,-1.039955,-0.314275,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
6,0.803975,0.373177,-0.481545,1.181090,-0.356983,0.178788,-0.096518,0.020373,0.965356,-0.314275,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
7,-0.523702,-0.424447,-0.787191,0.939071,-0.356983,1.094838,1.983921,-1.774818,-0.438361,1.643191,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
8,0.122032,-0.277210,-0.750804,-0.114351,-0.356983,-0.737261,-0.096518,0.020373,0.564294,-0.314275,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
9,0.384550,0.045396,-0.241394,2.154264,-0.356983,-0.737261,-0.096518,0.020373,1.366418,-0.314275,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [8]:
val_df['num__annual_inc'].sort_values(ascending=False)

209810    171.502584
406765    162.405975
192883    160.587745
8556      157.067757
413835    156.948010
             ...    
88738      -1.278408
393812     -1.300785
362141     -1.311156
367799     -1.332987
386438     -1.332987
Name: num__annual_inc, Length: 420204, dtype: float64

In [9]:
one_row = val_df.loc[[209810]]
one_row

,num__loan_amnt,num__installment,num__annual_inc,num__dti,num__delinq_2yrs,num__inq_last_6mths,num__mths_since_last_delinq,num__mths_since_last_record,num__open_acc,num__pub_rec,...,cat__addr_state_TX,cat__addr_state_UT,cat__addr_state_VA,cat__addr_state_VT,cat__addr_state_WA,cat__addr_state_WI,cat__addr_state_WV,cat__addr_state_WY,cat__initial_list_status_f,cat__initial_list_status_w
209810,1.169086,0.219205,171.502584,-2.177887,-0.356983,-0.737261,-0.096518,0.020373,0.163232,-0.314275,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [10]:
one_row.shape

(1, 147)

In [11]:
one_row.columns = (
    one_row.columns.astype(str)
    .str.replace(r"[\[\]]", "_", regex=True)
    .str.replace("<", "lt", regex=False)
)

In [12]:
explaination = explainer(one_row)

In [13]:
shap_df = pd.DataFrame({
    "feature": explaination.feature_names,
    'shap_value': explaination.values[0]
})

shap_df = shap_df.sort_values('shap_value', key=abs, ascending=False)

In [14]:
shap_df.head(5)

,feature,shap_value
58,cat__term_ 36 months,0.445120
2,num__annual_inc,-0.365052
57,num__fico_mid,-0.256253
3,num__dti,-0.153472
59,cat__term_ 60 months,0.110616


In [15]:
one_row['cat__term_ 60 months']

209810    1.0
Name: cat__term_ 60 months, dtype: float64

In [16]:
explaination.base_values

array([-1.6116569], dtype=float32)

In [17]:
sum(explaination.values[0])

np.float32(-0.93185043)

In [18]:
explaination.base_values + sum(explaination.values[0])

array([-2.5435073], dtype=float32)

In [19]:
raw_margin = model[1].predict(val_df.loc[[209810]], output_margin=True)
raw_margin

array([-2.5435088], dtype=float32)

In [20]:
1/(1 + np.exp(-explaination.base_values))

array([0.1663587], dtype=float32)

In [21]:
1/(1 + np.exp(-raw_margin))

array([0.07286378], dtype=float32)

In [22]:
y_val = feature_splits['val'][1]

In [23]:
y_val

,target
421094,0
397580,0
397581,1
397582,0
397583,0
...,...
29558,1
29557,0
29556,0
29564,0


In [24]:
one_row = val_df.loc[[29558]]
one_row.columns = (
    one_row.columns.astype(str)
    .str.replace(r"[\[\]]", "_", regex=True)
    .str.replace("<", "lt", regex=False)
)

In [25]:
explaination = explainer(one_row)

In [26]:
shap_df = pd.DataFrame({
    "feature": explaination.feature_names,
    'shap_value': explaination.values[0]
})

shap_df = shap_df.sort_values('shap_value', key=abs, ascending=False)

In [27]:
shap_df.head(5)

,feature,shap_value
19,num__acc_open_past_24mths,0.485492
3,num__dti,0.438596
47,num__num_tl_op_past_12m,0.245814
1,num__installment,-0.230954
58,cat__term_ 36 months,-0.200505


In [28]:
explaination.base_values + sum(explaination.values[0])

array([-0.78570336], dtype=float32)

In [29]:
raw_margin = model[1].predict(val_df.loc[[29558]], output_margin=True)
raw_margin

array([-0.7857027], dtype=float32)

In [30]:
1/(1 + np.exp(-explaination.base_values))

array([0.1663587], dtype=float32)

In [31]:
1/(1 + np.exp(-raw_margin))

array([0.31309214], dtype=float32)

In [32]:
shap_df.head().to_dict(orient='records')

[{'feature': 'num__acc_open_past_24mths', 'shap_value': 0.48549196124076843},
 {'feature': 'num__dti', 'shap_value': 0.43859612941741943},
 {'feature': 'num__num_tl_op_past_12m', 'shap_value': 0.2458137720823288},
 {'feature': 'num__installment', 'shap_value': -0.23095358908176422},
 {'feature': 'cat__term_ 36 months', 'shap_value': -0.200504869222641}]

In [33]:
from credit_risk.modeling.predict import predict_one

2026-07-20 18:58:55.607 | INFO     | credit_risk.modeling.predict:<module>:18 - Loading the preprocessor and tuned xgb model...
2026-07-20 18:58:55.617 | INFO     | credit_risk.modeling.predict:<module>:20 - Both loaded successfully!
2026-07-20 18:58:55.617 | INFO     | credit_risk.modeling.predict:<module>:22 - Building the col to onehot cols list map...
2026-07-20 18:58:55.619 | INFO     | credit_risk.modeling.predict:<module>:29 - CAT_TO_ONEHOT map built successfully!
2026-07-20 18:58:55.619 | INFO     | credit_risk.modeling.predict:<module>:31 - Creating the SHAP Explainer...
2026-07-20 18:58:55.768 | INFO     | credit_risk.modeling.predict:<module>:33 - SHAP explainer created successfully!
2026-07-20 18:58:55.768 | INFO     | credit_risk.modeling.predict:<module>:35 - Loading the threshold to make the prediction...
2026-07-20 18:58:55.770 | INFO     | credit_risk.modeling.predict:<module>:41 - Loaded successfully!


In [45]:
from credit_risk.dataset import load_splits, PROCESSED_DATA_DIR

In [46]:
_, val, _, _ = load_splits(PROCESSED_DATA_DIR / "after_eda")

2026-07-20 19:05:41.230 | INFO     | credit_risk.dataset:load_splits:263 - Checking if the files exists...
2026-07-20 19:05:41.231 | INFO     | credit_risk.dataset:load_splits:265 - Loading the Cached files...
2026-07-20 19:05:41.631 | INFO     | credit_risk.dataset:load_splits:273 - Loaded sucessfully all the splits and the metadata, Train_df shape: (466042, 110), val_df shape: (420204, 110), test_df shape: (431712, 110)


In [58]:
one_row = val.iloc[0].to_dict()

In [59]:
target = one_row['target']

In [60]:
del one_row['target']

In [61]:
pred, prob, reason_codes = predict_one(raw_input=one_row)

2026-07-20 19:14:21.470 | INFO     | credit_risk.modeling.predict:predict_one:51 - Adding the transformed features...
2026-07-20 19:14:21.473 | INFO     | credit_risk.modeling.predict:predict_one:53 - Adding the issue date as now...
2026-07-20 19:14:21.474 | INFO     | credit_risk.features:add_credit_yrs:170 - Inside Function: add_credit_yrs
2026-07-20 19:14:21.474 | INFO     | credit_risk.features:add_credit_yrs:172 - Adding the feature 'credit_yrs'
2026-07-20 19:14:21.477 | INFO     | credit_risk.features:add_credit_yrs:174 - 'credit_age_yrs added successfully!'
2026-07-20 19:14:21.477 | INFO     | credit_risk.features:add_fico_mid:179 - Inside Function: add_fico_mid
2026-07-20 19:14:21.477 | INFO     | credit_risk.features:add_fico_mid:181 - Adding the feature 'fico_mid'
2026-07-20 19:14:21.477 | INFO     | credit_risk.features:add_fico_mid:183 - 'fico_mid' added successfully!
2026-07-20 19:14:21.477 | INFO     | credit_risk.features:drop_columns:162 - Inside Function: drop_columns


In [63]:
pred, target

(0, 0)

In [64]:
prob

0.09350311756134033

In [65]:
reason_codes

{'num__installment': -0.2848484218120575,
 'cat__term': -0.26333168148994446,
 'num__dti': -0.15732090175151825,
 'num__total_bc_limit': 0.12151739001274109,
 'num__fico_mid': 0.1166466549038887}